In [ ]:
# 1. Install zstd dependency
!apt-get install -y zstd

# 2. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Start Ollama server bound to GPU
import os
import subprocess
import time

# Ensure any previous CPU instance is terminated
!pkill ollama

# Set environment variables so Ollama uses all available GPU layers
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["OLLAMA_NUM_GPU"] = "999"

# Launch Ollama server as a background process with GPU environment enabled
subprocess.Popen(["ollama", "serve"], env=os.environ, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Give the server a few seconds to initialize CUDA
print("Waiting for Ollama server to start on GPU...")
time.sleep(5)
print("Ollama GPU server is running!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 93 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Waiting for Ollama server to start on GPU...
Ollama GPU server is running!


In [ ]:
# 1. Install system-level tools (Tesseract OCR & PDF rendering engine)
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr poppler-utils > /dev/null

# 2. Install Python packages
!pip install -q pytesseract pdf2image ollama pillow

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
!ollama pull qwen2.5:3b

In [ ]:
# Cell 3
from google.colab import files
import os

print("Please upload a sample House BL document (PDF or Image):")
uploaded = files.upload()

# Get the filename of the uploaded file
file_name = list(uploaded.keys())[0]
print(f"File '{file_name}' successfully uploaded and ready for extraction.")

Please upload a sample House BL document (PDF or Image):


Saving Sample_HBL.pdf to Sample_HBL (1).pdf
File 'Sample_HBL (1).pdf' successfully uploaded and ready for extraction.


In [ ]:
import json
import time
import pytesseract
from pdf2image import convert_from_path
import ollama

def extract_hbl_number_fast(file_path):
    print(f"Processing '{file_path}'...")

    # 1. OCR Step (~0.5s)
    images = convert_from_path(file_path, first_page=1, last_page=1, dpi=150)
    text = pytesseract.image_to_string(images[0])

    # 2. Text LLM Step (~1.5s on GPU)
    prompt = f"""
    Extract the House Bill of Lading Number from this document text.
    Text:
    {text}

    Return ONLY JSON:
    {{"hbl_number": "string or null", "confidence": "high/medium/low"}}
    """

    response = ollama.chat(
    model='qwen2.5:3b',
    messages=[{'role': 'user', 'content': prompt}],
    options={
        'temperature': 0.0,
        'num_gpu': 999  # Explicitly pushes all model layers to GPU VRAM
    }
)

    return response['message']['content']

# Test execution time
start = time.time()
output = extract_hbl_number_fast(file_name)
print(output)
print(f"\nTime taken: {time.time() - start:.2f} seconds")

Processing 'Sample_HBL (1).pdf'...
```json
{
  "hbl_number": "HPS1201KOB-221",
  "confidence": "high"
}
```

Time taken: 11.34 seconds
